# QComponent - Interdigitated transmon qubit

> 💡 **Using this tutorial without the Qt GUI**
> 
> This tutorial uses the desktop `MetalGUI`. To follow along on Colab, Binder, JupyterHub, or any environment where Qt isn't available, **replace any `gui.rebuild()` / `gui.screenshot()` call with `qm.view(design)`** — it renders the design to a matplotlib `Figure` you can display inline or save with `fig.savefig(...)`.
> 
> See [1.4 Headless quick view](../1%20Overview/1.4%20Headless%20quick%20view%20%28no%20Qt%20GUI%29.ipynb) for a complete runnable walkthrough and [`docs/headless-usage.rst`](../../docs/headless-usage.rst) for the full reference.

This demo notebook goes over how to use the interdigitated transmon component, similar to those describedin Gambetta et. al., IEEE Trans. on Superconductivity Vol. 27, No. 1 (2007). 

#### Preparation
First, let's import the key libraries for qiskit metal: 

In [ ]:
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict  # , open_docs

(temporary patch) the creation of the cpw geomeptries from the interdigitated transmon pins, appear to cause a short-edge error rooted into the precision of the point alignment. For the timebeing you can run the following cell to suppress said violation.

In [ ]:
import qiskit_metal.toolbox_metal.math_and_overrides as mov

mov.DECIMAL_PRECISION = 7

#### Create the design
Next, let's fire up the GUI: 

In [ ]:
design = designs.DesignPlanar()
gui = MetalGUI(design)

The name of the component located in the qlibrary is "Transmon_Interdigitated" and we can take a look at the various input options:

In [ ]:
from qiskit_metal.qlibrary.qubits.Transmon_Interdigitated import TransmonInterdigitated

TransmonInterdigitated.get_template_options(design)

Now let's create three transmons, each centered at a specific (x,y) coordinate: 

In [ ]:
from qiskit_metal.qlibrary.qubits.Transmon_Interdigitated import TransmonInterdigitated

design.overwrite_enabled = True
q1 = TransmonInterdigitated(
    design, "qubit1", options=dict(pos_x="-2.0mm", orientation="-90")
)
q2 = TransmonInterdigitated(
    design, "qubit2", options=dict(pos_x="2.0mm", orientation="90")
)
q3 = TransmonInterdigitated(
    design, "qubit3", options=dict(pos_y="3.0mm", orientation="180")
)
gui.rebuild()
gui.autoscale()

To check that the qpin connections are functioning correctly, we'll connect these transmons with CPWs. Let's first connect qpin2 of "qubit1" to qpin3 of "qubit3". We can do this using the RouteMeander functionality. 

In [ ]:
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander

In [ ]:
ops = dict(fillet="90um")

In [ ]:
options = Dict(
    total_length="5mm",
    pin_inputs=Dict(
        start_pin=Dict(component="qubit1", pin="bus1"),
        end_pin=Dict(component="qubit3", pin="bus2"),
    ),
    lead=Dict(start_straight="0.5mm", end_straight="0.0mm"),
    meander=Dict(asymmetry="0mm"),
    **ops,
)

# Below I am creating a CPW without assigning its name.
#  Therefore running this cell twice will create two CPW's instead of overwriting the previous one
#  To prevent that we add the cpw.delete() statement.
#  The try-except wrapping is needed to suppress errors during the first run of this cell
try:
    cpw.delete()
except NameError:
    pass

cpw1 = RouteMeander(design, options=options)
gui.rebuild()
gui.autoscale()

Now let's connect qubit 2 to qubit 3:

In [ ]:
options = Dict(
    total_length="5mm",
    pin_inputs=Dict(
        start_pin=Dict(component="qubit2", pin="bus2"),
        end_pin=Dict(component="qubit3", pin="bus1"),
    ),
    lead=Dict(start_straight="0.5mm", end_straight="0.0mm"),
    meander=Dict(asymmetry="0mm"),
    **ops,
)

# try:
#    cpw.delete()
# except NameError: pass

cpw2 = RouteMeander(design, options=options)
gui.rebuild()
gui.autoscale()

And finally, let's connect pin3 of qubit1 to pin2 of qubit 2: 

In [ ]:
options = Dict(
    total_length="8mm",
    pin_inputs=Dict(
        start_pin=Dict(component="qubit1", pin="bus2"),
        end_pin=Dict(component="qubit2", pin="bus1"),
    ),
    lead=Dict(start_straight="0.5mm", end_straight="0.5mm"),
    meander=Dict(asymmetry="0mm"),
    **ops,
)

# try:
#    cpw.delete()
# except NameError: pass

cpw2 = RouteMeander(design, options=options)
gui.rebuild()
gui.autoscale()